# Pipelines & Data Leakage

In this notebook, we will learn why preprocessing must be performed correctly.

We will compare:

1. A manually managed preprocessing workflow.
2. A Pipeline-based workflow.

The goal is to understand how Pipelines help make ML workflows safer and reproducible.

In [1]:
from sklearn.datasets import load_iris
import pandas as pd

data = load_iris(as_frame=True)

df = data.frame

X = df.drop(columns="target")
y = df["target"]

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Correct Manual Preprocessing

If we manually scale the data, the scaler must be fitted only on the training data.

The test data is transformed using the scaler that was already fitted on the training data.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

model.fit(X_train_scaled, y_train)

manual_accuracy = model.score(
    X_test_scaled,
    y_test,
)

print(f"Manual preprocessing accuracy: {manual_accuracy:.4f}")

Manual preprocessing accuracy: 0.9333


# Pipeline

Now we combine the scaler and model into a Pipeline.

The Pipeline handles the sequence:

StandardScaler → LogisticRegression

The Pipeline can then be treated like a single estimator.

In [6]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

pipeline.fit(X_train, y_train)

pipeline_accuracy = pipeline.score(
    X_test,
    y_test,
)

print(f"Pipeline accuracy: {pipeline_accuracy:.4f}")

Pipeline accuracy: 0.9333


# Why the Pipeline Matters

The important benefit is not that the Pipeline necessarily produces a different accuracy.

The important benefit is that preprocessing and model training are connected into one workflow.

This becomes especially important when using cross-validation and hyperparameter tuning.

The Pipeline ensures that preprocessing is fitted within the appropriate training portion rather than accidentally using validation data.


In [7]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
)

print(f"Mean CV accuracy: {cv_scores.mean():.4f}")
print(f"Std CV accuracy: {cv_scores.std():.4f}")

Mean CV accuracy: 0.9583
Std CV accuracy: 0.0264


# Pipeline + Hyperparameter Tuning

A Pipeline can also be passed directly to GridSearchCV.

This allows preprocessing and model hyperparameters to be evaluated together without manually preprocessing the entire dataset first.

This is the pattern we used earlier with SVM.

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

svm_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", SVC()),
    ]
)

param_grid = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["linear", "rbf"],
}

search = GridSearchCV(
    svm_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)

print(f"\nBest CV accuracy: {search.best_score_:.4f}")

Best parameters:
{'model__C': 0.1, 'model__kernel': 'linear'}

Best CV accuracy: 0.9750


# The Data Leakage Rule

The most important rule from this notebook:

> Learn preprocessing parameters only from training data.

Examples include:

- StandardScaler
- MinMaxScaler
- SimpleImputer
- Feature selection
- PCA

When these operations are placed inside a Pipeline, they can be fitted correctly within the training portion of each cross-validation split.

# Interview Insight

### What is data leakage?

Using information that should not be available during model training or model selection.

### Why is data leakage dangerous?

It can produce unrealistically good evaluation results that do not generalize to unseen data.

### Why use a Pipeline?

To combine preprocessing and the estimator into one workflow and reduce the risk of inconsistent preprocessing and leakage.

### Should we fit a scaler on the complete dataset before cross-validation?

No.

The scaler should be fitted only on the training portion of each fold.

### Can a Pipeline be used with GridSearchCV?

Yes.

This allows preprocessing and model parameters to be evaluated together safely.

# Notebook Summary

In this notebook, we learned:

- What data leakage is.
- Why preprocessing must be fitted only on training data.
- How to correctly preprocess train and test data manually.
- How to use a Pipeline.
- Why Pipelines are important for cross-validation.
- Why Pipelines are important for hyperparameter tuning.

# What's Next?

Next is the final foundation notebook:

## 40 — End-to-End Machine Learning Workflow

We will connect everything we have learned into one complete workflow.

After Notebook 40, we stop adding fundamental notebooks and move into the real project.